# Fase 14 — Statistik urutan token pada citra validasi NYATA (tanpa training)

**Tujuan.** Menutup `[VERIFY: input used for the order statistics]` di §5.5 naskah. Klaim §5.5: pada checkpoint P1-fix
(seed 1), urutan token di tiga peta beresolusi tertinggi (25.600 / 6.400 / 1.600 token pada masukan 640 × 640)
**identik dengan urutan raster**. Angka lama dihitung pada *masukan sintetis* (catatan serah-terima); urutan token
bergantung masukan → dihitung ulang di sini pada **682 citra validasi steel-pipe yang sebenarnya**.

**Yang diukur** (tiap checkpoint × tiap blok deformable × tiap citra × tiap presisi):
- `moved_frac` — fraksi token yang posisinya berbeda dari urutan raster (0 = identik raster); `max_disp` — pergeseran maks.
- `frac_strict_desc` — fraksi pasangan token bertetangga (urutan raster) yang posisi-jalurnya TURUN = penataan-ulang
  SUNGGUHAN; `frac_ties` — fraksi pasangan bertetangga yang posisi-jalurnya SAMA PERSIS (seri pembulatan, mis. fp16).
  `monotone` = tidak ada penurunan → urutan = raster *sampai dengan seri* (beda urutan hanya dari seri).
- `frac_sat` (|tanh Δp| > 0,99), `frac_oob` (posisi jalur di luar [−1, 1]); `bias_dp` = bias keluaran offset urutan
  (parameter, tidak bergantung masukan).
- Tiga presisi: **fp32**; **amp** (bobot fp32 + autocast fp16 = numerik training); **half** (`model.half()` = validasi Ultralytics).

**Checkpoint:** `output/p1-fix_seed1_150ep_lr0.002/weights/best.pt` dan `output/p1-fix_seed3_150ep_lr0.002/weights/best.pt`
(identitas dicek dari `train_args` di dalam checkpoint: nama run & seed; sha256 berkas dicatat).
**Masukan:** 682 citra `val2021`, letterbox 640 × 640 (persegi, `auto=False`, tanpa upscale), RGB, /255 — sama dgn pra-proses
validasi Ultralytics kecuali bentuk persegi (validasi memakai batch *rect*; persegi dipilih agar jumlah token = klaim naskah).

**Keluaran (Drive `analysis/`):** `fase14_urutan_token_per_citra.csv`, `fase14_urutan_token_ringkas.csv`,
`fase14_bias_offset.csv`, `fase14_meta.json`. Kirim keempatnya ke Claude.

**Biaya:** ±10–15 menit A100 (unduh dataset + kernel dari cache Drive + 2 checkpoint × 3 presisi × 682 citra).
Runtime dilepas otomatis di akhir (sel terakhir) dan bila ada sel yang gagal (Sel 1).

**Aturan.** Notebook ini TIDAK melatih apa pun dan TIDAK menulis ke folder `output/`. Sel 0–4c disalin apa adanya dari
notebook Fase 13 (sudah teruji); sel 3, 5, 6, 7 baru (diuji offline di CPU dgn checkpoint & citra palsu).

## 0. Pin Lingkungan — torch 2.11.0+cu128 (WAJIB paling awal, sebelum `import torch`)

Colab kini memasang torch **2.11.0+cu130** (CUDA 13). Kernel `selective_scan` gagal di-compile di CUDA 13, dan
run P1-fix/Fase 10–12 sebelumnya memakai **2.11.0+cu128**. Sel ini memasang build cu128 (versi torch SAMA, beda
build CUDA) lalu me-RESTART runtime otomatis. **Setelah restart: Runtime → Run all SEKALI LAGI** — kali kedua sel ini
hanya mencetak "OK" dan notebook lanjut, memakai wheel `selective_scan` cu128 yang sudah ada di cache Drive.


In [1]:
import importlib.metadata as _md, subprocess, sys, os
TORCH_PIN = "2.11.0+cu128"          # = lingkungan run P1-fix / Fase 10-12 (tracker: torch 2.11.0+cu128, py3.13)
_tv = _md.version("torch")
print("torch terpasang:", _tv)
if _tv != TORCH_PIN:
    assert "torch" not in sys.modules, "torch sudah ter-import di sesi ini -- Runtime > Restart session, lalu Run all"
    _vis = "0.26.0"   # pasangan RESMI torch 2.11.0 (metadata wheel: Requires-Dist torch==2.11.0) -- TIDAK
                      # diambil dari torchvision bawaan Colab (bisa sudah versi lain bila Colab naik ke torch 2.12+)
    _pkgs = [f"torch=={TORCH_PIN}", f"torchvision=={_vis}+cu128"]
    print("Memasang", _pkgs, "dari indeks PyTorch cu128 (sekali per sesi, beberapa menit)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs,
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    print("Terpasang:", subprocess.run([sys.executable, "-c", "import torch,torchvision;print(torch.__version__, torch.version.cuda, torchvision.__version__)"],
                                       capture_output=True, text=True).stdout.strip())
    print("\n>>> Runtime akan RESTART sekarang. Setelah tersambung lagi: Runtime > Run all SEKALI LAGI. <<<")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    print(f"OK: torch = {TORCH_PIN} -- lanjut.")


torch terpasang: 2.11.0+cu128
OK: torch = 2.11.0+cu128 -- lanjut.


## 1. Pengaman Biaya + Mount Drive + Kredensial Kaggle (Cache)

**Pengaman biaya** dipasang PALING AWAL: bila ada sel yang gagal saat Run all (termasuk mount Drive / kaggle.json), error ditampilkan lalu runtime dilepas otomatis (jeda 60 detik; tekan Stop untuk menahan).


In [2]:
# ---- PENGAMAN BIAYA: bila ada sel GAGAL (exception) saat "Run all", tampilkan error lalu lepas runtime ----
# Tanpa ini, satu error sebelum/sesudah training membuat GPU menganggur (tetap ditagih) sampai Colab memutus sendiri.
# Menekan tombol Stop (KeyboardInterrupt) TIDAK memicu pengaman ini. Ada jeda 60 detik: tekan Stop di jeda itu bila
# ingin menahan runtime. Set False bila sedang men-debug interaktif.
AUTO_UNASSIGN_ON_ERROR = True

def _lepas_runtime_bila_error(shell, etype, evalue, tb, tb_offset=None):
    stb = shell.InteractiveTB.structured_traceback(etype, evalue, tb, tb_offset=tb_offset)
    shell._showtraceback(etype, evalue, stb)          # tampilkan error-nya dulu
    if shell.user_ns.get("AUTO_UNASSIGN_ON_ERROR", True):
        print("\n⛔ Sel GAGAL -- runtime dilepas otomatis dalam 60 detik agar kuota Colab tidak terbuang.\n"
              "   Checkpoint & results.csv yang sudah ada aman di Drive; perbaiki penyebabnya lalu jalankan ulang notebook.\n"
              "   (Tekan Stop sekarang bila ingin menahan runtime.)", flush=True)
        import time
        try:
            time.sleep(60)
        except KeyboardInterrupt:   # Stop ditekan di jeda -> runtime DITAHAN, pengaman tetap terpasang
            print("Dibatalkan (Stop): runtime TIDAK dilepas. Pengaman tetap aktif untuk sel berikutnya.", flush=True)
            return None
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as _e:
            print(f"Gagal melepas runtime otomatis ({_e}) -- matikan manual: Runtime > Disconnect and delete runtime")
    return None

get_ipython().set_custom_exc((Exception,), _lepas_runtime_bila_error)
print("Pengaman biaya aktif: runtime dilepas otomatis bila ada sel yang gagal.")

from google.colab import drive
drive.mount('/content/drive')

import os, stat, shutil

DRIVE_ROOT = "/content/drive/MyDrive/demamba-yolowd"
os.makedirs(DRIVE_ROOT, exist_ok=True)
KAGGLE_JSON_CACHE = f"{DRIVE_ROOT}/kaggle.json"

os.makedirs("/root/.kaggle", exist_ok=True)
assert os.path.exists(KAGGLE_JSON_CACHE), (
    f"kaggle.json tidak ada di {KAGGLE_JSON_CACHE} -- jalankan fase-setup-colab-mamba.ipynb dulu."
)
shutil.copy(KAGGLE_JSON_CACHE, "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", stat.S_IRUSR | stat.S_IWUSR)
print("kaggle.json dimuat dari cache Drive.")

Pengaman biaya aktif: runtime dilepas otomatis bila ada sel yang gagal.
Mounted at /content/drive
kaggle.json dimuat dari cache Drive.


## 2. Tarik Dataset Steel-pipe + deform_ss2d.py dari Kaggle + Tripwire (sha256 & isi dataset) + Deteksi GPU


In [3]:
import subprocess, sys, glob, torch

# TF32 dimatikan (konsisten dgn semua run proyek sejak B2 seed 1).
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print(f"TF32 dimatikan (matmul={torch.backends.cuda.matmul.allow_tf32}, cudnn={torch.backends.cudnn.allow_tf32})")

# ---------- Cek GPU PALING AWAL (sebelum unduhan) -- GPU salah = berhenti dalam hitungan detik ----------
_gpu = torch.cuda.get_device_name(0)
print(f"GPU: {_gpu}  (compute cap {torch.cuda.get_device_capability(0)})")
# Semua run grid-fix sebelumnya (P1 penuh steel-pipe, SWRD v1) memakai A100. GPU lain = numerik & waktu berbeda
# (T4 ±3x lebih lambat -> biaya naik). Ubah ke True HANYA bila sengaja, dan catat di tracker.
ALLOW_NON_A100 = False
assert "A100" in _gpu or ALLOW_NON_A100, (
    f"GPU = {_gpu}, BUKAN A100. Runtime > Change runtime type > A100, lalu Run all lagi "
    f"(atau set ALLOW_NON_A100 = True dengan sengaja).")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
os.makedirs("/content/data", exist_ok=True)
os.makedirs("/content/deform_src", exist_ok=True)

STEEL_DIR = "/content/data/steelpipe"
if os.path.exists(STEEL_DIR):
    shutil.rmtree(STEEL_DIR)   # selalu unduh bersih
os.makedirs(STEEL_DIR)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/dsf-yolo-steel-tube-dataset",
                "-p", STEEL_DIR, "--unzip"], check=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "benyxu/deform-ss2d-py",
                "-p", "/content/deform_src", "--unzip"], check=True)

DEFORM_SRC = glob.glob("/content/deform_src/**/deform_ss2d.py", recursive=True)[0]

# ---------- TRIPWIRE VERSI deform_ss2d.py (sha256) -- HARUS grid-fix ----------
#   cfed78de317a7781... = PRA-perbaikan (grid +0.5 px)   -> DITOLAK
#   675445871a2e4d55... = GRID-FIX                        -> DIPAKAI DI SINI (= P1 penuh steel-pipe & SWRD v1)
#   51520b33f9f77f28... = v2 jalur-terbatas (Fase 12)     -> DITOLAK
import hashlib
DEFORM_SHA256_EXPECTED = "675445871a2e4d55400175ef5f6fda83c90ece651bf818e3f54481bc14a6d9aa"
_raw = open(DEFORM_SRC, "rb").read()
DEFORM_SHA256 = hashlib.sha256(_raw).hexdigest()
_norm = hashlib.sha256(_raw.replace(b"\r\n", b"\n")).hexdigest()
_txt = _raw.decode("utf-8", "replace")
print(f"deform_ss2d.py sha256 = {DEFORM_SHA256}")
if "linspace(0.5, H - 0.5" in _txt or "linspace(0.5, N - 0.5" in _txt:
    raise RuntimeError("deform_ss2d.py = versi PRA-PERBAIKAN (grid +0.5 px). Unggah versi grid-fix ke Kaggle 'deform-ss2d-py'.")
if "bounded_path_positions" in _txt:
    raise RuntimeError("deform_ss2d.py = versi v2 jalur-terbatas (Fase 12). Bukan yang dipakai di sini.")
assert DEFORM_SHA256 == DEFORM_SHA256_EXPECTED or _norm == DEFORM_SHA256_EXPECTED, (
    f"deform_ss2d.py BUKAN versi grid-fix (sha256 {DEFORM_SHA256[:16]}..., harus 675445871a2e4d55...). STOP.")
DEFORM_SHA256 = DEFORM_SHA256_EXPECTED   # yang dicatat di CSV = sha berkas SUMBER (sebelum patch runtime NoProj)
print("Tripwire versi: deform_ss2d.py = GRID-FIX (675445871a2e4d55...) -- OK")

# ---------- Dataset steel-pipe: cari root YOLO + TRIPWIRE isi ----------
# Root YOLO = folder yang punya images/train2021 DAN labels/train2021 (= .../steel-tube-dataset-all/yolo, sama dgn
# fase-0-clean). Bila rilis memuat format lain dgn nama split sama, folder 'yolo' didahulukan.
_cands = sorted((p for p in glob.glob(f"{STEEL_DIR}/**/images/train2021", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(os.path.dirname(p)), "labels", "train2021"))),
                key=lambda p: ("/yolo/" not in p, p.count("/"), p))
assert _cands, f"Folder images/train2021 + labels/train2021 tidak ditemukan di {STEEL_DIR} -- cek hasil unduhan Kaggle"
if len(_cands) > 1:
    print("  (beberapa kandidat root dataset; dipilih yang pertama):", _cands)
STEEL_BASE = os.path.dirname(os.path.dirname(_cands[0]))
print(f"STEEL_BASE = {STEEL_BASE}")
STEEL_NAMES = ["air-hole", "bite-edge", "broken-arc", "crack", "dense-gas-hole", "non-fusion", "slag-inclusion", "unfused-hole"]
_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
EXPECTED_STEEL = {"train2021": 2726, "val2021": 682}   # jumlah citra (rilis Yang et al. 2021, diverifikasi Sesi 3)
_tot_obj, _val_obj = 0, 0
for _sp, _n in EXPECTED_STEEL.items():
    _imgs = [p for p in glob.glob(f"{STEEL_BASE}/images/{_sp}/*") if os.path.splitext(p)[1].lower() in _IMG_EXT]
    _stems = {os.path.splitext(os.path.basename(p))[0] for p in _imgs}
    _txt_all = glob.glob(f"{STEEL_BASE}/labels/{_sp}/*.txt")
    _labs = [p for p in _txt_all if os.path.splitext(os.path.basename(p))[0] in _stems]   # abaikan classes.txt dsb.
    if len(_txt_all) != len(_labs):
        print(f"  ({_sp}: {len(_txt_all) - len(_labs)} berkas .txt tanpa citra pasangan diabaikan, mis. classes.txt)")
    assert len(_imgs) == _n, f"{_sp}: {len(_imgs)} citra, harus {_n}"
    _k = 0
    for _p in _labs:
        for _l in open(_p).read().splitlines():
            if _l.strip():
                _r = _l.split()
                assert len(_r) == 5 and 0 <= int(_r[0]) < 8, f"label rusak: {_p}"
                _k += 1
    _tot_obj += _k
    if _sp == "val2021":
        _val_obj = _k
    print(f"Tripwire {_sp}: {len(_imgs)} citra, {len(_labs)} berkas label, {_k} objek")
assert _tot_obj == 6783 and _val_obj == 1381, f"objek total {_tot_obj} (harus 6783), val {_val_obj} (harus 1381)"
for _c in glob.glob(f"{STEEL_BASE}/labels/*.cache"):
    os.remove(_c)   # cache label lama dari unduhan -- dibuat ulang oleh Ultralytics
print("Tripwire dataset steel-pipe: COCOK (2.726 + 682 citra, 6.783 objek, 1.381 di val)")


TF32 dimatikan (matmul=False, cudnn=False)
GPU: NVIDIA A100-SXM4-40GB  (compute cap (8, 0))
deform_ss2d.py sha256 = 675445871a2e4d55400175ef5f6fda83c90ece651bf818e3f54481bc14a6d9aa
Tripwire versi: deform_ss2d.py = GRID-FIX (675445871a2e4d55...) -- OK
STEEL_BASE = /content/data/steelpipe/steel-tube-dataset-all/yolo
Tripwire train2021: 2726 citra, 2726 berkas label, 5402 objek
Tripwire val2021: 682 citra, 682 berkas label, 1381 objek
Tripwire dataset steel-pipe: COCOK (2.726 + 682 citra, 6.783 objek, 1.381 di val)


## 3. Konfigurasi — checkpoint P1-fix seed 1 & 3 (tidak perlu diubah)

In [4]:
# ============================================================
#  KONFIGURASI (tidak perlu diubah)
# ============================================================
MODEL = "P1"                                   # P1-fix = DeformSS2D penuh, capacity-matched, kode GRID-FIX
CONFIG_YAML, EXPECTED_PARAMS, EXPECTED_NDEF = "DeMamba-YOLO-T.yaml", 5_930_869, 10
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
CKPTS = {   # seed -> best.pt (path dari penulis, 24 Okt 2026)
    1: f"{DRIVE_OUTPUT}/p1-fix_seed1_150ep_lr0.002/weights/best.pt",
    3: f"{DRIVE_OUTPUT}/p1-fix_seed3_150ep_lr0.002/weights/best.pt",
}
IMGSZ, BATCH = 640, 8
MODES = ("fp32", "amp", "half")
EXPECTED_N_VAL = 682
DEVICE = "cuda:0"
OUT_DIR = f"{DRIVE_ROOT}/analysis"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PER_IMG = f"{OUT_DIR}/fase14_urutan_token_per_citra.csv"
OUT_SUMMARY = f"{OUT_DIR}/fase14_urutan_token_ringkas.csv"
OUT_BIAS    = f"{OUT_DIR}/fase14_bias_offset.csv"
OUT_META    = f"{OUT_DIR}/fase14_meta.json"
for _s, _p in CKPTS.items():
    assert os.path.exists(_p), f"best.pt seed {_s} tidak ditemukan: {_p}"
    print(f"seed {_s}: {_p}  ({os.path.getsize(_p)/1e6:.1f} MB)")
print(f"IMGSZ={IMGSZ}  BATCH={BATCH}  MODES={MODES}  -> {OUT_DIR}")

seed 1: /content/drive/MyDrive/demamba-yolowd/output/p1-fix_seed1_150ep_lr0.002/weights/best.pt  (12.2 MB)
seed 3: /content/drive/MyDrive/demamba-yolowd/output/p1-fix_seed3_150ep_lr0.002/weights/best.pt  (12.2 MB)
IMGSZ=640  BATCH=8  MODES=('fp32', 'amp', 'half')  -> /content/drive/MyDrive/demamba-yolowd/analysis


## 4a. Clone Mamba-YOLO + Catat Commit


In [5]:
import re, py_compile

REPO = "/content/Mamba-YOLO"
# WAJIB sebelum rmtree: pastikan cwd proses TIDAK sedang di dalam REPO (bisa terjadi kalau
# notebook di-run ulang di kernel yang sama setelah sel pre-flight sempat os.chdir(REPO) --
# menghapus folder yang sedang "ditempati" proses bikin SEMUA subprocess berikutnya gagal
# dgn "fatal: Unable to read current working directory").
os.chdir("/content")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HZAI-ZJNU/Mamba-YOLO.git", REPO], check=True)

DEFORM = f"{REPO}/ultralytics/nn/modules/deform_ss2d.py"
shutil.copy(DEFORM_SRC, DEFORM)

src = open(DEFORM).read()
checks = {
    "STE (gradien Delta-p)": "x_soft" in src,
    "ssm_ratio=1.4 (capacity match P1)": "ssm_ratio=1.4" in src,
    "subclass block terdaftar": "class DeformVSSBlock" in src and "class DeformXSSBlock" in src,
}
if not all(checks.values()):
    raise RuntimeError("deform_ss2d.py BUKAN versi final -- cek dataset 'deform-ss2d-py' di Kaggle.")
print("Tripwire deform_ss2d.py OK")

REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
REPO_COMMIT_KNOWN = "51f7be9bd5ab907b6f7d2c91bb71ca5a4be063aa"   # commit HZAI-ZJNU/Mamba-YOLO yang tercatat (21 Okt 2026)
print(f"Mamba-YOLO commit: {REPO_COMMIT}")
if REPO_COMMIT != REPO_COMMIT_KNOWN:
    print("⚠️  Commit upstream BERBEDA dari yang tercatat -- patch di Sel 4b tetap dicek ketat (assert 1x); "
          "catat commit ini di tracker sebelum melanjutkan.")


Tripwire deform_ss2d.py OK
Mamba-YOLO commit: 51f7be9bd5ab907b6f7d2c91bb71ca5a4be063aa


## 4b. Patch Registrasi + Patch P1-NoProj + YAML Semua Varian + YAML Dataset

Patch registrasi identik notebook SWRD v1/Fase 2, ditambah NP-1…NP-4 (`PATCH-p1-noproj.md`) yang diterapkan ke salinan grid-fix, dan patch 6.1 (`np.trapz` → `np.trapezoid`, numpy ≥ 2.4). Sel ini tidak memakai GPU/jaringan (diuji offline sebelum dikirim).


In [6]:
import re, py_compile
def patch(path, subs):
    p = f"{REPO}/{path}"
    src = open(p).read()
    for old, new, tag in subs:
        if new in src:
            print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new)
        print(f"  OK {tag}")
    open(p, "w").write(src)

print("\n### Patch")
patch("ultralytics/nn/modules/mamba_yolo.py", [
    ("class XSSBlock(nn.Module):\n    def __init__(",
     "class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.1"),
    ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,",
     "nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,", "1.2"),
    ("class VSSBlock(nn.Module):\n    def __init__(",
     "class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(", "1.3"),
    ("            self.op = SS2D(",
     "            self.op = self.SSM_CLASS(", "1.4"),
])
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("from .mamba_yolo import SS2D",
     "from .mamba_yolo import SS2D, VSSBlock, XSSBlock", "2.1"),
])

# ---- Patch NP-1..NP-5: varian P1-NoProj (PATCH-p1-noproj.md), diterapkan RUNTIME ke salinan grid-fix ----
# Semua argumen baru default ke perilaku lama -> P1 & 5 varian parsial TIDAK berubah (dicek di pre-flight Sel 5:
# P1 tetap 5.930.869 params). sha256 yang dicatat di CSV = berkas sumber grid-fix SEBELUM patch ini.
patch("ultralytics/nn/modules/deform_ss2d.py", [
    ("    def __init__(self, in_ch: int, kernel_size: int):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1)\n",
     "    def __init__(self, in_ch: int, kernel_size: int, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        pad = kernel_size // 2 if kernel_size > 1 else 0\n"
     "        self.use_proj = use_proj\n"
     "        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1) if use_proj else nn.Identity()\n", "NP-1"),
    ("    def __init__(self, embed_dim: int, kernel_size: int = 5):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size)\n",
     "    def __init__(self, embed_dim: int, kernel_size: int = 5, use_proj: bool = True):\n"
     "        super().__init__()\n"
     "        self.conv_offset = ConvOffset(embed_dim, kernel_size, use_proj=use_proj)\n", "NP-2"),
    ("                 deform_kernel_size: int = None,\n"
     "                 **kwargs):\n",
     "                 deform_kernel_size: int = None,\n"
     "                 deform_use_proj: bool = True,\n"
     "                 **kwargs):\n", "NP-3a"),
    ("        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks) for _ in range(n_deform)\n"
     "        ])\n",
     "        self.deform_use_proj = deform_use_proj\n"
     "        self.deform_scans = nn.ModuleList([\n"
     "            DeformableScan(d_inner, kernel_size=ks, use_proj=deform_use_proj) for _ in range(n_deform)\n"
     "        ])\n", "NP-3b"),
    ('\nif __name__ == "__main__":',
     '\n\nclass DeformVSSBlockNP(VSSBlock):\n'
     '    """P1-NoProj (backbone): ssm_ratio TETAP dari YAML/VSSBlock (2.0) + ConvOffset tanpa proj.\n'
     '    Total model 6.329.230 params (+5,72% vs B2) -- sengaja di atas toleransi +/-5% (stress test)."""\n'
     '    SSM_CLASS = partial(DeformSS2D, deform_mode="k3_replace", deform_kernel_size=5, deform_use_proj=False)\n'
     '    # TIDAK meng-override __init__ (beda dgn DeformVSSBlock yg memaksa ssm_ratio=1.4)\n'
     '\n\nclass DeformXSSBlockNP(XSSBlock):\n'
     '    """P1-NoProj (neck). Lihat DeformVSSBlockNP."""\n'
     '    SSM_CLASS = partial(DeformSS2D, deform_mode="k3_replace", deform_kernel_size=5, deform_use_proj=False)\n'
     '\n\nif __name__ == "__main__":', "NP-4"),
])
patch("ultralytics/nn/tasks.py", [
    ("    XSSBlock\n)\n",
     "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n", "3.1"),
    ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
     "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock, DeformVSSBlockNP, DeformXSSBlockNP\n        }:", "3.2"),
    ("RepC3, XSSBlock}:",
     "RepC3, XSSBlock, DeformXSSBlock, DeformXSSBlockNP}:", "3.3"),
    ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
     '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:', "4.1"),
    ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
     '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)', "4.2"),
])
patch("ultralytics/utils/torch_utils.py", [
    ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
     '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:', "5.1"),
])

y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
y = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", y)
y = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml", "w").write(y)

# Ablasi B: 2 varian campuran -- backbone-saja-deformable, neck-saja-deformable.
# Split di penanda "# Mamba-YOLO PAFPN" (awal section head/neck), regex-swap CUMA
# di satu bagian. Params sudah diverifikasi offline: BackboneOnly=5.952M(-0.58%),
# NeckOnly=5.966M(-0.36%) vs B2 -- keduanya dlm toleransi +-5% TANPA ssm_ratio baru.
base_yaml = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
split_marker = "# Mamba-YOLO PAFPN"
idx = base_yaml.find(split_marker)
assert idx != -1, "penanda split YAML tidak ketemu -- struktur Mamba-YOLO-T.yaml berubah?"
backbone_part, head_part = base_yaml[:idx], base_yaml[idx:]
assert "VSSBlock" in backbone_part and "XSSBlock" not in backbone_part
assert "XSSBlock" in head_part and "VSSBlock" not in head_part

backbone_deform = re.sub(r"\bVSSBlock\b", "DeformVSSBlock", backbone_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-backbone-only.yaml", "w").write(
    backbone_deform + head_part)

head_deform = re.sub(r"\bXSSBlock\b", "DeformXSSBlock", head_part)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-neck-only.yaml", "w").write(
    backbone_part + head_deform)
print("  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate")

# Ablasi B-lanjut: 3 varian stage-wise di backbone (literatur-deformable-lintas-paradigma.md
# SS6.3). Neck TIDAK disentuh (tetap XSSBlock biasa di semua 3 varian ini) -- isolasi bersih
# dari pertanyaan Ablasi B sebelumnya. Tiap baris backbone punya komentar posisi unik
# ("# 1","# 3","# 5","# 7") dipakai utk menarget baris tepat, assert-guarded (fail loud
# kalau match != 1x, mis. krn upstream ubah format YAML).
def _swap_positions(text, positions):
    lines = text.split("\n")
    hit = {p: 0 for p in positions}
    for i, line in enumerate(lines):
        stripped = line.rstrip()
        for p in positions:
            if stripped.endswith(f"# {p}") and "VSSBlock" in line and "Deform" not in line:
                lines[i] = line.replace("VSSBlock", "DeformVSSBlock")
                hit[p] += 1
    for p in positions:
        assert hit[p] == 1, f"posisi {p}: ditemukan {hit[p]}x (harus tepat 1x) -- format YAML berubah?"
    return "\n".join(lines)

stagewise_variants = {"bb-deep": [7], "bb-late": [5, 7], "bb-early": [1, 3]}
for vname, positions in stagewise_variants.items():
    result = _swap_positions(base_yaml, positions)
    open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-{vname}.yaml", "w").write(result)
print(f"  OK {len(stagewise_variants)} YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate")

# BB-Late-Alt (BERSYARAT, backlog Section 2d) -- sumbu kerapatan intra-stage. Posisi 5
# dipecah jadi 3 baris terpisah (D-S-D) krn parse_model bikin n INSTANCE TERPISAH dari
# kelas yg sama per baris (bukan loop internal) -- lihat tasks.py baris ~946. Memecah 1
# baris jadi 3 menggeser SEMUA indeks "from" eksplisit di bawahnya +2 -- WAJIB diperbaiki,
# assert-guarded (kalau salah, parse_model gagal keras karena channel Concat tak cocok,
# bukan diam-diam salah). Sudah diverifikasi offline: save-layers hasil parse_model =
# [3,7,10,13,16,19,22], cocok persis perhitungan manual.
_lines = base_yaml.split("\n")
_t5 = None
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 5") and "VSSBlock" in _l and "Deform" not in _l:
        assert _t5 is None, "posisi 5: >1 baris cocok"
        _t5 = _i
assert _t5 is not None, "posisi 5 tidak ditemukan"
_dsd = [
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5a (D)",
    "  - [-1, 1, VSSBlock, [512]]             # 5b (S)",
    "  - [-1, 1, DeformVSSBlock, [512]]      # 5c (D)",
]
_lines = _lines[:_t5] + _dsd + _lines[_t5+1:]
_hit7 = 0
for _i, _l in enumerate(_lines):
    if _l.rstrip().endswith("# 7") and "VSSBlock" in _l and "Deform" not in _l:
        _lines[_i] = _l.replace("VSSBlock", "DeformVSSBlock")
        _hit7 += 1
assert _hit7 == 1, f"posisi 7: {_hit7}x"
_text = "\n".join(_lines)
_reindex = [
    ("  - [[-1, 5], 1, Concat, [1]]  # cat backbone P4",
     "  - [[-1, 7], 1, Concat, [1]]  # cat backbone P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 11], 1, Concat, [1]]  # cat head P4",
     "  - [[-1, 13], 1, Concat, [1]]  # cat head P4 (geser +2 krn split posisi 5)"),
    ("  - [[-1, 8], 1, Concat, [1]]  # cat head P5",
     "  - [[-1, 10], 1, Concat, [1]]  # cat head P5 (geser +2 krn split posisi 5)"),
    ("  - [[14, 17, 20], 1, Detect, [nc]]  # Detect(P3, P4, P5)",
     "  - [[16, 19, 22], 1, Detect, [nc]]  # Detect(P3, P4, P5) (geser +2 krn split posisi 5)"),
]
for _old, _new in _reindex:
    assert _text.count(_old) == 1, f"GAGAL cari saat reindex: {_old!r}"
    _text = _text.replace(_old, _new)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-bb-late-alt.yaml", "w").write(_text)
print("  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki")

# ---- Patch 6.1: np.trapz DIHAPUS mulai numpy 2.4.0; Ultralytics 8.2.29 memakainya utk menghitung AP ----
# Tanpa patch ini, bila Colab sudah memakai numpy >= 2.4, training CRASH di validasi epoch pertama.
# np.trapezoid = fungsi yang SAMA (nama baru sejak numpy 2.0; np.trapz di 2.0-2.3 hanya memanggil np.trapezoid)
# -> angka mAP IDENTIK dgn run sebelumnya. Fallback np.trapz hanya utk numpy 1.x.
patch("ultralytics/utils/metrics.py", [
    ("ap = np.trapz(np.interp(x, mrec, mpre), x)",
     "ap = (np.trapezoid if hasattr(np, 'trapezoid') else np.trapz)(np.interp(x, mrec, mpre), x)", "6.1"),
])
import numpy as _np
print(f"  numpy {_np.__version__}: np.trapz {'ada' if hasattr(_np, 'trapz') else 'TIDAK ADA (patch 6.1 mencegah crash)'}")

# P1-NoProj: posisi deformable SAMA dgn P1 (1,3,5,7,11,14,17,20), kelas *NP (ssm_ratio 2.0, tanpa proj)
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T.yaml").read()
assert _y.count("DeformVSSBlock") == 4 and _y.count("DeformXSSBlock") == 4, "YAML P1 tak terduga"
_y = re.sub(r"\bDeformVSSBlock\b", "DeformVSSBlockNP", _y)
_y = re.sub(r"\bDeformXSSBlock\b", "DeformXSSBlockNP", _y)
open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/DeMamba-YOLO-T-noproj.yaml", "w").write(_y)
print("  OK YAML P1-NoProj (DeMamba-YOLO-T-noproj.yaml) digenerate")

# data YAML steel-pipe dgn path ABSOLUT (nama kelas = nama lama; pemetaan naskah ada di Sel 0)
import yaml as _yaml
STEEL_YAML = f"{REPO}/ultralytics/cfg/datasets/steelpipe.yaml"
with open(STEEL_YAML, "w") as _f:
    _yaml.safe_dump({"path": STEEL_BASE, "train": "images/train2021", "val": "images/val2021",
                     "nc": 8, "names": STEEL_NAMES}, _f, sort_keys=False, allow_unicode=True)
print(f"  OK steelpipe.yaml ditulis: {STEEL_YAML}")
print(open(STEEL_YAML).read())

for f in ["ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py",
          "ultralytics/nn/tasks.py", "ultralytics/utils/torch_utils.py", "ultralytics/utils/metrics.py"]:
    py_compile.compile(f"{REPO}/{f}", doraise=True)
print("  OK sintaks semua file valid -- SEMUA PATCH + YAML OK")



### Patch
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
  ~ 2.1: skip
  OK NP-1
  OK NP-2
  OK NP-3a
  OK NP-3b
  OK NP-4
  OK 3.1
  OK 3.2
  OK 3.3
  OK 4.1
  OK 4.2
  OK 5.1
  OK 2 YAML Ablasi B (backbone-only, neck-only) berhasil digenerate
  OK 3 YAML Ablasi B-lanjut (bb-deep, bb-late, bb-early) berhasil digenerate
  OK 1 YAML BB-Late-Alt (D-S-D, bersyarat) berhasil digenerate, indeks head diperbaiki
  OK 6.1
  numpy 2.1.3: np.trapz ada
  OK YAML P1-NoProj (DeMamba-YOLO-T-noproj.yaml) digenerate
  OK steelpipe.yaml ditulis: /content/Mamba-YOLO/ultralytics/cfg/datasets/steelpipe.yaml
path: /content/data/steelpipe/steel-tube-dataset-all/yolo
train: images/train2021
val: images/val2021
nc: 8
names:
- air-hole
- bite-edge
- broken-arc
- crack
- dense-gas-hole
- non-fusion
- slag-inclusion
- unfused-hole

  OK sintaks semua file valid -- SEMUA PATCH + YAML OK


## 4c. Build Kernel `selective_scan` + Install


In [7]:
import re
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja"], check=True)
nproc = int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip() or "2")
os.environ["MAX_JOBS"] = str(max(1, nproc))

# Cache wheel selective_scan ke Drive. KUNCI CACHE WAJIB memuat versi CUDA torch:
# torch "2.11.0" bisa dibangun utk CUDA 12 (cu128) maupun CUDA 13 (cu130) dgn nomor versi SAMA.
# Kunci lama hanya "torch2.11.0" -> wheel hasil compile CUDA 12 (sesi lalu) dipasang ke torch CUDA 13
# -> ImportError libcudart.so.12. Kunci baru memisahkan keduanya; wheel lama tetap di Drive (tak dipakai).
import shutil as _sh
cap = torch.cuda.get_device_capability(0)
_tv = torch.__version__.replace("+", "-")
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{_tv}_cuda{torch.version.cuda}_py{sys.version_info.major}{sys.version_info.minor}"
SS_WHEEL_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"
os.makedirs(SS_WHEEL_DIR, exist_ok=True)

def _nvcc_version():
    nvcc = os.path.join(os.environ.get("CUDA_HOME", "/usr/local/cuda"), "bin", "nvcc")
    if not os.path.exists(nvcc):
        nvcc = _sh.which("nvcc") or nvcc
    try:
        out = subprocess.run([nvcc, "--version"], capture_output=True, text=True).stdout
        return re.search(r"release (\d+)\.(\d+)", out).group(0).split()[-1]
    except Exception:
        return None

def _ss_import_ok():
    # dicek di proses TERPISAH: import yang gagal tidak meninggalkan modul rusak di kernel ini
    r = subprocess.run([sys.executable, "-c", "import torch, selective_scan_cuda_core"], capture_output=True, text=True)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

def _ss_install(wheels):
    # --force-reinstall --no-deps: nama & versi wheel sama dgn yg (mungkin) sudah terpasang -> tanpa ini pip melewatinya
    subprocess.run([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps"] + wheels, check=True)

print(f"torch {torch.__version__} | CUDA torch {torch.version.cuda} | nvcc {_nvcc_version()} | {SS_TAG}")
cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
# Wheel LAMA (kunci tanpa versi CUDA, mis. 'sm80_torch2.11.0_py313') dibuat di sesi torch 2.11.0+cu128.
# Hanya dipakai bila torch sekarang juga build cu128, dan tetap WAJIB lolos uji import di bawah.
_legacy = f"{DRIVE_ROOT}/wheels/selective_scan_sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
if not cached_ss and torch.__version__.endswith("+cu128") and glob.glob(f"{_legacy}/*.whl"):
    for _w in glob.glob(f"{_legacy}/*.whl"):
        _sh.copy(_w, SS_WHEEL_DIR)
    cached_ss = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    print(f"Wheel lama ditemukan di {os.path.basename(_legacy)} (cu128) -- disalin ke kunci baru, akan diuji import")
if cached_ss:
    print(f"Wheel selective_scan cache ditemukan ({SS_TAG}) -- install langsung, skip compile")
    _ss_install(cached_ss)
    ok, err = _ss_import_ok()
    if not ok:
        print(f"  Wheel cache GAGAL di-import ({err}) -- dihapus, compile ulang")
        for w in cached_ss: os.remove(w)
        cached_ss = []
if not cached_ss:
    _nv = _nvcc_version()
    assert _nv is not None, "nvcc tidak ditemukan -- tidak bisa compile selective_scan."
    assert _nv.split(".")[0] == str(torch.version.cuda).split(".")[0], (
        f"CUDA MAJOR berbeda: torch dibangun utk CUDA {torch.version.cuda}, nvcc sistem {_nv}. "
        f"Tidak ada wheel cache utk GPU ini ({SS_TAG}). Pakai GPU A100 (sama dgn run sebelumnya) atau kirim pesan ini ke Claude.")
    # CUDA 13 menghapus dukungan arsitektur < sm_75 (Maxwell/Pascal/Volta). setup.py selective_scan
    # (warisan VMamba) meng-compile sm_70 -> nvcc 13 menolak 'compute_70'. Ganti ke sm_75 HANYA bila CUDA>=13.
    # Tidak mengubah numerik di A100: kernel yg dipakai GPU sm_80 tetap cubin sm_80. (sm_75 = T4.)
    _setup = f"{REPO}/selective_scan/setup.py"
    if int(str(torch.version.cuda).split(".")[0]) >= 13:
        _ss = open(_setup).read()
        _n70 = len(re.findall(r"compute_70\s*,\s*code\s*=\s*sm_70", _ss))
        if _n70:
            _ss = re.sub(r"compute_70\s*,\s*code\s*=\s*sm_70", "compute_75,code=sm_75", _ss)
            open(_setup, "w").write(_ss)
        print(f"  setup.py selective_scan: {_n70} target sm_70 diganti sm_75 (CUDA {torch.version.cuda}); "
              f"arch tersisa: {sorted(set(re.findall(r'sm_(\d+)', _ss)))}")
    print(f"Belum ada cache utk {SS_TAG} -- compile sekali ini (~beberapa menit), lalu disimpan ke Drive")
    subprocess.run([sys.executable, "-m", "pip", "wheel", ".", "--no-deps", "-v", "-w", SS_WHEEL_DIR],
                   cwd=f"{REPO}/selective_scan", check=True)
    new_ss_wheels = glob.glob(f"{SS_WHEEL_DIR}/*.whl")
    _ss_install(new_ss_wheels)
    ok, err = _ss_import_ok()
    assert ok, f"selective_scan hasil compile tetap gagal di-import: {err}"
    print(f"Tersimpan ke {SS_WHEEL_DIR} -- sesi berikutnya dgn kombinasi GPU/torch/CUDA yg sama langsung cepat")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)

for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]:
    del sys.modules[k]
import selective_scan_cuda_core
print("selective_scan_cuda_core berhasil di-import.")
ENV_INFO = dict(repo_commit=REPO_COMMIT, torch=torch.__version__, cuda=torch.version.cuda, nvcc=_nvcc_version(), python=sys.version.split()[0],
                gpu=torch.cuda.get_device_name(0), ss_tag=SS_TAG)
print("Lingkungan run (catat di tracker):", ENV_INFO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seaborn", "thop", "timm", "einops"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                "wandb", "ray", "tensorboard", "tensorboardX", "tensorboard-data-server",
                "clearml", "comet_ml", "dvclive", "mlflow", "neptune"])

print("\nSEMUA PATCH + BUILD OK.")


torch 2.11.0+cu128 | CUDA torch 12.8 | nvcc 13.0 | sm80_torch2.11.0-cu128_cuda12.8_py313
Wheel selective_scan cache ditemukan (sm80_torch2.11.0-cu128_cuda12.8_py313) -- install langsung, skip compile
selective_scan_cuda_core berhasil di-import.
Lingkungan run (catat di tracker): {'repo_commit': '51f7be9bd5ab907b6f7d2c91bb71ca5a4be063aa', 'torch': '2.11.0+cu128', 'cuda': '12.8', 'nvcc': '13.0', 'python': '3.13.16', 'gpu': 'NVIDIA A100-SXM4-40GB', 'ss_tag': 'sm80_torch2.11.0-cu128_cuda12.8_py313'}

SEMUA PATCH + BUILD OK.


## 5. Pre-flight — arsitektur P1 (5.930.869 params, 10 DeformSS2D)

In [8]:
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from ultralytics import settings
settings.update({"tensorboard": False})
from ultralytics.nn.tasks import DetectionModel
from ultralytics.nn.modules.deform_ss2d import DeformSS2D, DeformableScan, ConvOffset
import torch.nn as nn

_m = DetectionModel(f"ultralytics/cfg/models/mamba-yolo/{CONFIG_YAML}", ch=3, nc=8, verbose=False)
_np = sum(x.numel() for x in _m.parameters())
_nd = sum(1 for x in _m.modules() if isinstance(x, DeformSS2D))
_nproj = sum(1 for x in _m.modules() if isinstance(x, ConvOffset) and not isinstance(x.proj, nn.Identity))
print(f"P1 (YAML): params {_np:,} (target {EXPECTED_PARAMS:,})  DeformSS2D {_nd}  ConvOffset ber-proj {_nproj}")
assert (_np, _nd, _nproj) == (EXPECTED_PARAMS, EXPECTED_NDEF, 10), "Pre-flight GAGAL -- STOP."
del _m
print("Pre-flight LOLOS.")

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/c

P1 (YAML): params 5,930,869 (target 5,930,869)  DeformSS2D 10  ConvOffset ber-proj 10
Pre-flight LOLOS.


## 6. Statistik urutan token — 2 checkpoint × 3 presisi × 682 citra val (tanpa training)

In [9]:
# ---- Sel 6: statistik urutan token per blok deformable, per citra, per presisi ----
import copy, hashlib, json, cv2, numpy as np, pandas as pd
from ultralytics.data.augment import LetterBox

dev = torch.device(DEVICE)
VAL_DIR = f"{STEEL_BASE}/images/val2021"
val_imgs = sorted(p for p in glob.glob(f"{VAL_DIR}/*") if os.path.splitext(p)[1].lower() in {".jpg", ".jpeg", ".png", ".bmp"})
assert len(val_imgs) == EXPECTED_N_VAL, f"{len(val_imgs)} citra val, harus {EXPECTED_N_VAL}"
_lb = LetterBox(new_shape=(IMGSZ, IMGSZ), auto=False, scaleup=False)

def load_batch(paths):
    ims = []
    for p in paths:
        im = cv2.imread(p)
        assert im is not None, f"gagal baca {p}"
        im = _lb(image=im)
        ims.append(np.ascontiguousarray(im[..., ::-1].transpose(2, 0, 1)))   # BGR->RGB, HWC->CHW
    return torch.from_numpy(np.stack(ims)).float().div_(255.0)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def load_ckpt(path, seed):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    ta = dict(ck.get("train_args") or {})
    run_dir = os.path.basename(os.path.dirname(os.path.dirname(path)))
    assert ta.get("seed") == seed, f"train_args seed = {ta.get('seed')} != {seed} ({path})"
    assert str(ta.get("name", "")) == run_dir, f"train_args name = {ta.get('name')!r} != folder {run_dir!r}"
    m = ck.get("ema") or ck["model"]
    m = m.float().eval()
    n_par = sum(x.numel() for x in m.parameters())
    n_def = sum(1 for x in m.modules() if isinstance(x, DeformSS2D))
    assert (n_par, n_def) == (EXPECTED_PARAMS, EXPECTED_NDEF), f"model di checkpoint: {n_par:,} params, {n_def} DeformSS2D"
    print(f"  seed {seed}: name={ta.get('name')}  optimizer={ta.get('optimizer')}  lr0={ta.get('lr0')}  "
          f"epochs={ta.get('epochs')}  params={n_par:,}  DeformSS2D={n_def}")
    return m, ta

ROWS, BIAS, META = [], [], {"imgsz": IMGSZ, "n_val": len(val_imgs), "modes": list(MODES), "ckpt": {}}
CUR = {}

def _hook(name):
    def h(mod, inp, out):
        x, idx = inp[0], out[1]
        B, _, H, W = x.shape
        N = H * W
        with torch.no_grad():
            off = mod.conv_offset(x)                                   # dtype/konteks SAMA dgn forward aslinya
            tdp = off[:, 2:3].tanh().flatten(1)
            pp = tdp + mod._path_ref(N, B, x.dtype, x.device)        # = path_pos di DeformableScan.forward
            ar = torch.arange(N, device=idx.device)
            moved = (idx != ar).float().mean(1)
            maxd = (idx - ar).abs().max(1).values
            d = pp[:, 1:] - pp[:, :-1]
            desc = (d < 0).float().mean(1)
            ties = (d == 0).float().mean(1)
            sat = (tdp.abs() > 0.99).float().mean(1)
            oob = (pp.abs() > 1).float().mean(1)
            mabs = tdp.abs().float().mean(1)
            same = (torch.argsort(pp, dim=-1) == idx).all(1)          # cek: hitung ulang = urutan yg dipakai model
        st = torch.stack([moved, maxd.float(), desc, ties, sat, oob, mabs, same.float()], 1).cpu().numpy()
        for b in range(B):
            ROWS.append(dict(seed=CUR["seed"], mode=CUR["mode"], module=name, H=H, W=W, N=N,
                             img=os.path.basename(CUR["paths"][b]), dtype=str(pp.dtype).replace("torch.", ""),
                             moved_frac=st[b, 0], max_disp=int(st[b, 1]), identical=bool(st[b, 0] == 0),
                             frac_strict_desc=st[b, 2], monotone=bool(st[b, 2] == 0), frac_ties=st[b, 3],
                             frac_sat=st[b, 4], frac_oob=st[b, 5], mean_abs_tanh_dp=st[b, 6], recompute_match=bool(st[b, 7])))
    return h

for seed, path in CKPTS.items():
    print(f"=== seed {seed} ===")
    base, ta = load_ckpt(path, seed)
    META["ckpt"][seed] = {"path": path, "sha256": sha256_file(path), "name": ta.get("name"), "seed": ta.get("seed"),
                          "optimizer": ta.get("optimizer"), "lr0": ta.get("lr0"), "epochs": ta.get("epochs")}
    for name, mod in base.named_modules():
        if isinstance(mod, DeformableScan):
            w, b = mod.conv_offset.out.weight.detach().float(), mod.conv_offset.out.bias.detach().float()
            BIAS.append(dict(seed=seed, module=name, bias_dp=b[2].item(), w_dp_norm=w[2].norm().item(),
                             bias_dy=b[0].item(), bias_dx=b[1].item()))
    for mode in MODES:
        m = copy.deepcopy(base).to(dev)
        m = m.half() if mode == "half" else m.float()
        m.eval()
        hs = [mod.register_forward_hook(_hook(n)) for n, mod in m.named_modules() if isinstance(mod, DeformableScan)]
        assert len(hs) == EXPECTED_NDEF
        CUR.update(seed=seed, mode=mode)
        for i in range(0, len(val_imgs), BATCH):
            paths = val_imgs[i:i + BATCH]
            CUR["paths"] = paths
            xb = load_batch(paths).to(dev)
            xb = xb.half() if mode == "half" else xb
            with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=(mode == "amp")):
                m(xb)
        for hk in hs:
            hk.remove()
        del m
        if dev.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  {mode}: selesai ({len(val_imgs)} citra)")
    del base

df = pd.DataFrame(ROWS)
bias = pd.DataFrame(BIAS)
assert len(df) == len(CKPTS) * len(MODES) * EXPECTED_NDEF * len(val_imgs), f"jumlah baris {len(df)} tak terduga"
df.to_csv(OUT_PER_IMG, index=False)
bias.to_csv(OUT_BIAS, index=False)
META.update(deform_sha256=DEFORM_SHA256, env=ENV_INFO if "ENV_INFO" in globals() else None,
            preprocess="LetterBox(640x640, auto=False, scaleup=False), BGR->RGB, /255",
            recompute_match_all=bool(df["recompute_match"].all()))
json.dump(META, open(OUT_META, "w"), indent=2, default=str)
print(f"\nTersimpan: {OUT_PER_IMG} ({len(df):,} baris), {OUT_BIAS}, {OUT_META}")
print("Hitung-ulang urutan == urutan yang dipakai model di SEMUA baris:", META["recompute_match_all"])

=== seed 1 ===
  seed 1: name=p1-fix_seed1_150ep_lr0.002  optimizer=SGD  lr0=0.002  epochs=150  params=5,930,869  DeformSS2D=10
  fp32: selesai (682 citra)
  amp: selesai (682 citra)
  half: selesai (682 citra)
=== seed 3 ===
  seed 3: name=p1-fix_seed3_150ep_lr0.002  optimizer=SGD  lr0=0.002  epochs=150  params=5,930,869  DeformSS2D=10
  fp32: selesai (682 citra)
  amp: selesai (682 citra)
  half: selesai (682 citra)

Tersimpan: /content/drive/MyDrive/demamba-yolowd/analysis/fase14_urutan_token_per_citra.csv (40,920 baris), /content/drive/MyDrive/demamba-yolowd/analysis/fase14_bias_offset.csv, /content/drive/MyDrive/demamba-yolowd/analysis/fase14_meta.json
Hitung-ulang urutan == urutan yang dipakai model di SEMUA baris: True


## 7. Ringkasan + jawaban langsung utk klaim §5.5

In [10]:
# ---- Sel 7: ringkasan per checkpoint x presisi x blok + jawaban langsung utk klaim §5.5 ----
g = df.groupby(["seed", "mode", "module", "H", "W", "N"], sort=False)
summ = g.agg(n_img=("img", "count"), frac_img_identical=("identical", "mean"), frac_img_monotone=("monotone", "mean"),
             moved_frac_mean=("moved_frac", "mean"), moved_frac_max=("moved_frac", "max"), max_disp_max=("max_disp", "max"),
             strict_desc_max=("frac_strict_desc", "max"), ties_mean=("frac_ties", "mean"), sat_mean=("frac_sat", "mean"),
             oob_mean=("frac_oob", "mean")).reset_index()
summ = summ.merge(bias, on=["seed", "module"], how="left")
summ.to_csv(OUT_SUMMARY, index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
for (s, mo), t in summ.groupby(["seed", "mode"], sort=False):
    print(f"\n=== seed {s} | {mo} ===")
    print(t[["module", "H", "W", "N", "bias_dp", "frac_img_identical", "frac_img_monotone", "moved_frac_max",
             "max_disp_max", "strict_desc_max", "ties_mean", "sat_mean", "oob_mean"]].to_string(index=False))

print("\n=== Jawaban langsung (klaim §5.5) ===")
for s in CKPTS:
    b = bias[bias.seed == s]
    print(f"seed {s}: |bias_dp| >= 1.5 di {(b.bias_dp.abs() >= 1.5).sum()} dari {len(b)} blok "
          f"(rentang {b.bias_dp.min():+.2f} .. {b.bias_dp.max():+.2f})")
top_n = [int(x) for x in sorted(df.N.unique(), reverse=True)[:3]]
for s in CKPTS:
    for mo in MODES:
        t = summ[(summ.seed == s) & (summ["mode"] == mo) & (summ.N.isin(top_n))]
        print(f"seed {s} {mo:>4}: {len(t)} blok di N={top_n} | identik raster di SEMUA citra: "
              f"{int((t.frac_img_identical == 1).sum())}/{len(t)} blok | raster s/d seri (monoton) di semua citra: "
              f"{int((t.frac_img_monotone == 1).sum())}/{len(t)} blok | moved_frac maks {t.moved_frac_max.max():.4f} | "
              f"max_disp maks {int(t.max_disp_max.max())}")
print(f"\nTersimpan: {OUT_SUMMARY}")


=== seed 1 | fp32 ===
                        module   H   W     N   bias_dp  frac_img_identical  frac_img_monotone  moved_frac_max  max_disp_max  strict_desc_max  ties_mean  sat_mean  oob_mean
     model.1.op.deform_scans.0 160 160 25600  5.378906            1.000000           1.000000        0.000000             0         0.000000   0.000000  1.000000  0.499961
     model.3.op.deform_scans.0  80  80  6400 -3.609375            0.998534           0.998534        0.000312             1         0.000156   0.000000  1.000000  0.499586
   model.5.0.op.deform_scans.0  40  40  1600  3.500000            1.000000           1.000000        0.000000             0         0.000000   0.000000  1.000000  0.499589
   model.5.1.op.deform_scans.0  40  40  1600  3.738281            1.000000           1.000000        0.000000             0         0.000000   0.000000  1.000000  0.500000
   model.5.2.op.deform_scans.0  40  40  1600 -2.341797            0.000000           0.000000        0.159375        

## Checklist

- [ ] Sel 0: torch = 2.11.0+cu128 (sesi baru: Run all → restart otomatis → Run all lagi)
- [ ] Sel 1: "Pengaman biaya aktif" tercetak
- [ ] Sel 2: GPU = A100, tripwire `deform_ss2d.py` = GRID-FIX, dataset steel-pipe COCOK (682 citra val)
- [ ] Sel 3: kedua `best.pt` ditemukan
- [ ] Sel 5: Pre-flight LOLOS (5.930.869 params, 10 DeformSS2D)
- [ ] Sel 6: identitas checkpoint (nama run & seed dari `train_args`) cocok; "Hitung-ulang urutan == … : True"
- [ ] Sel 7: ringkasan tercetak
- [ ] Kirim ke Claude: `fase14_urutan_token_per_citra.csv`, `fase14_urutan_token_ringkas.csv`, `fase14_bias_offset.csv`,
      `fase14_meta.json` (Drive `demamba-yolowd/analysis/`) + salinan keluaran Sel 6–7

## 10. Matikan Runtime (Wajib Dijalankan Terakhir)

In [11]:
# Matikan runtime Colab otomatis setelah semua sel di atas selesai --
# supaya TIDAK terus memakan kuota Colab Pro kalau kamu tidak sempat
# matikan manual. Ini API resmi Google, benar-benar melepas VM-nya.
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)  # jeda singkat, jaga-jaga ada file yang masih proses ditulis ke Drive

try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")


Semua proses selesai. Runtime akan di-disconnect otomatis...
